<a href="https://colab.research.google.com/github/cazador505/Spectral/blob/master/PyGMTSAR_Flooding_Block_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Inundaciones en la región Lambayeque (2017) usando Coherencia InSAR (CI)


## Instalacion de PyGMTSAR

Instala PyGMTSAR y librerias GMTSAR binarias (incluyendo SNAPHU)

In [ ]:
import platform, sys, os
if 'google.colab' in sys.modules:
    !{sys.executable} -m pip install -q pygmtsar
    import importlib.resources as resources
    with resources.as_file(resources.files('pygmtsar.data') / 'google_colab.sh') as google_colab_script_filename:
        !sh {google_colab_script_filename}

    from google.colab import output
    output.enable_custom_widget_manager()

    import xvfbwrapper
    display = xvfbwrapper.Xvfb(width=800, height=600)
    display.start()

# especifica ruta de installation GMTSAR
PATH = os.environ['PATH']
if PATH.find('GMTSAR') == -1:
    PATH = os.environ['PATH'] + ':/usr/local/GMTSAR/bin/'
    %env PATH {PATH}

# vicualiza version de PyGMTSAR
from pygmtsar import __version__
__version__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 65.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)




update-alternatives: using /usr/bin/gcc-9 to provide /usr/bin/gcc (gcc) in auto mode
There is only one alternative in link group gcc (providing /usr/bin/gcc): /usr/bin/gcc-9
Nothing

'2025.4.8.post1'

## Carga e Inicializa modulos Python

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import json
from dask.distributed import Client
import dask

import os
import asf_search as asf
from pygmtsar import ASF
from pygmtsar import S1
from pygmtsar import Stack, tqdm_dask, Tiles, XYZTiles

In [ ]:
# plotting modules
import pyvista as pv
# magic trick for white background
pv.set_plot_theme("document")
import panel
panel.extension(comms='ipywidgets')
panel.extension('vtk')
from contextlib import contextmanager
import matplotlib.pyplot as plt
@contextmanager
def mpl_settings(settings):
    original_settings = {k: plt.rcParams[k] for k in settings}
    plt.rcParams.update(settings)
    yield
    plt.rcParams.update(original_settings)
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.titlesize'] = 24
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
%matplotlib inline

In [ ]:
# define Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

In [ ]:
# =============================================================================
# CONFIGURACIÓN
# =============================================================================

USERNAME = "jorge_nole"
PASSWORD = "Jorge5317366"

In [ ]:
# =============================================================================
# DIRECTORIOS
# =============================================================================

WORKDIR = 'raw_kal'
DATADIR = 'data_kal'

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(DATADIR, exist_ok=True)


In [ ]:
# =============================================================================
# PARÁMETROS GLOBALES
# =============================================================================

GEOCODE_STEP = 45.          # resolución geocoding (px en grados/metros según proyección)
COARSEN = (3, 12)          # multilooking SAR <-- CORREGIDO (antes: 3,12)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# AOI
# =========================================================
aoi = gpd.read_file("/content/drive/MyDrive/Inundacion_GEE/PROVINCIA.shp")

if aoi.crs != "EPSG:4326":
    aoi = aoi.to_crs("EPSG:4326")

aoi_geom = aoi.geometry.union_all()

def inside_aoi(footprint):
    try:
        if footprint is None:
            return True
        return shape(footprint).intersects(aoi_geom)
    except:
        return True


In [ ]:
# =============================================================================
# FECHAS
# =============================================================================

START_DATE = '2017-02-01'
END_DATE   = '2017-04-06'

# =============================================================================
# PARÁMETROS InSAR
# =============================================================================
MIN_BASELINE = 3
MAX_TEMPORAL_BASELINE = 12

# =============================================================================
# PARÁMETROS SBAS ROBUSTO
# =============================================================================

MIN_BASELINE = 3


# BUSCAR ESCENAS SLC
# =============================================================================

print('\nBuscando imágenes Sentinel-1 SLC...\n')

# Convert aoi_geom to WKT for asf.search
AOI_WKT = aoi_geom.wkt

results = asf.search(
    platform=asf.PLATFORM.SENTINEL1,
    processingLevel='SLC',
    beamMode='IW',
    start=START_DATE,
    end=END_DATE,
    intersectsWith=AOI_WKT
)

print(f'Total escenas encontradas: {len(results)}')


In [ ]:
# =============================================================================
# EXTRAER METADATOS
# =============================================================================

data = []

for r in results:

    props = r.properties

    try:

        data.append({

            'sceneName'       : props['sceneName'],
            'startTime'       : props['startTime'],
            'flightDirection' : props['flightDirection'],
            'pathNumber'      : props['pathNumber'],
            'frameNumber'     : props.get('frameNumber', None),
            'url'             : r.geojson()['properties']['url']

        })

    except:
        pass

# =============================================================================
# DATAFRAME
# =============================================================================

df = pd.DataFrame(data)

# =============================================================================
# VALIDAR RESULTADOS
# =============================================================================

if len(df) == 0:

    raise Exception(
        'No se encontraron escenas SLC'
    )

# =============================================================================
# FECHAS
# =============================================================================

df['date'] = pd.to_datetime(df['startTime'])

# =============================================================================
# ORDENAR
# =============================================================================

df = df.sort_values('date')

# =============================================================================
# MOSTRAR ESCENAS S1B_IW_SLC__1SDV_20170327T105332_20170327T105359_004893_0088CB_534B'
# =============================================================================

print('\nESCENAS ENCONTRADAS:\n')

print(
    df[
        [
            'sceneName',
            'date',
            'flightDirection',
            'pathNumber',
            'frameNumber'
        ]
    ]
)


In [ ]:
# =============================================================================
# GENERAR PARES INTERFEROMÉTRICOS
# =============================================================================

pairs = []

# =============================================================================
# RECORRER ESCENAS
# =============================================================================

for i in range(len(df)): # Iterate through all scenes as master

    master = df.iloc[i]

    for j in range(i + 1, len(df)): # Nested loop to compare with all subsequent scenes
        slave  = df.iloc[j]

        # -------------------------------------------------------------
        # MISMA ÓRBITA
        # -------------------------------------------------------------

        same_orbit = (
            master['pathNumber'] == slave['pathNumber']
        )

        # -------------------------------------------------------------
        # MISMA DIRECCIÓN
        # -------------------------------------------------------------

        same_direction = (
            master['flightDirection'] == slave['flightDirection']
        )

        # -------------------------------------------------------------
        # MISMO FRAME
        # -------------------------------------------------------------

        same_frame = (
            master['frameNumber'] == slave['frameNumber']
        )

        # -------------------------------------------------------------
        # BASELINE TEMPORAL
        # -------------------------------------------------------------

        temporal_baseline = abs(
            (slave['date'] - master['date']).days
        )

        valid_time = (
            3 <= temporal_baseline <= MAX_TEMPORAL_BASELINE
        )
        # -------------------------------------------------------------
        # VALIDAR PAR
        # -------------------------------------------------------------

        if (
            same_orbit and
            same_direction and
            same_frame and
            valid_time
        ):

            pairs.append({

                'master' : master['sceneName'],
                'slave'  : slave['sceneName'],

                'master_date' : master['date'],
                'slave_date'  : slave['date'],

                'days' : temporal_baseline,

                'pathNumber' : master['pathNumber'],
                'frameNumber': master['frameNumber'],
                'direction'  : master['flightDirection']

            })

# =============================================================================
# DATAFRAME DE PARES
# =============================================================================

pairs_df = pd.DataFrame(pairs)

# =============================================================================
# PRINT PARES
# =============================================================================

print('\n======================================')
print('PARES INTERFEROMÉTRICOS')
print('======================================\n')

# Check if pairs_df is not empty before attempting to print its columns
if not pairs_df.empty:
    print(
        pairs_df[
            [
                'master',
                'slave',
                'days',
                'pathNumber',
                'frameNumber',
                'direction'
            ]
        ]
    )
else:
    print("No se encontraron pares interferométricos válidos.")

# =============================================================================
# GENERAR TRÍADAS
# =============================================================================

triads = []

print('\n======================================')
print('TRÍADAS INTERFEROMÉTRICAS')
print('======================================\n')

# Only attempt to generate triads if pairs_df is not empty
if not pairs_df.empty:
    for i in range(len(pairs_df)): # Iterate over all pairs as the first pair

        pair1 = pairs_df.iloc[i]

        for j in range(len(pairs_df)): # Iterate over all pairs as the second pair

            pair2 = pairs_df.iloc[j]

            # -------------------------------------------------------------
            # EVITAR MISMO PAR
            # -------------------------------------------------------------

            if i == j:
                continue

            # -------------------------------------------------------------
            # CONECTIVIDAD
            # -------------------------------------------------------------

            connected = (
                pair1['slave'] == pair2['master']
            )

            # -------------------------------------------------------------
            # MISMA GEOMETRÍA
            # -------------------------------------------------------------

            same_path = (
                pair1['pathNumber'] == pair2['pathNumber']
            )

            same_frame = (
                pair1['frameNumber'] == pair2['frameNumber']
            )

            same_direction = (
                pair1['direction'] == pair2['direction']
            )

            # -------------------------------------------------------------
            # VALIDAR TRÍADA
            # -------------------------------------------------------------

            if (
                connected and
                same_path and
                same_frame and
                same_direction
            ):

                triads.append({

                    'A' : pair1['master'],
                    'B' : pair1['slave'],
                    'C' : pair2['slave'],

                    'A_date' : pair1['master_date'],
                    'B_date' : pair1['slave_date'],
                    'C_date' : pair2['slave_date'],

                    'AB_days' : pair1['days'],
                    'BC_days' : pair2['days'],

                    'pathNumber' : pair1['pathNumber'],
                    'frameNumber': pair1['frameNumber'],
                    'direction'  : pair1['direction']

                })

                # ---------------------------------------------------------
                # PRINT
                # ---------------------------------------------------------

                print('--------------------------------------')
                print('TRÍADA VÁLIDA')
                print('--------------------------------------\n')

                print(f'A : {pair1["master"]}')
                print(f'Fecha A : {pair1["master_date"]}\n')

                print(f'B : {pair1["slave"]}')
                print(f'Fecha B : {pair1["slave_date"]}\n')

                print(f'C : {pair2["slave"]}')
                print(f'Fecha C : {pair2["slave_date"]}\n')

                print(f'A-B : {pair1["days"]} días')
                print(f'B-C : {pair2["days"]} días\n')

                print(f'Path      : {pair1["pathNumber"]}')
                print(f'Frame     : {pair1["frameNumber"]}')
                print(f'Direction : {pair1["direction"]}')

                print('\n--------------------------------------\n')
    if not triads:
        print("No se encontraron tríadas interferométricas válidas.")
else:
    print("No se pueden generar tríadas porque no se encontraron pares válidos.")

# =============================================================================
# DATAFRAME FINAL
# =============================================================================

triads_df = pd.DataFrame(triads)

# =============================================================================
# RESUMEN
# =============================================================================

print('\n======================================')
print('RESUMEN')
print('======================================')

print(f'Total pares   : {len(pairs_df)}')
print(f'Total tríadas : {len(triads_df)}')


In [ ]:
import pandas as pd
import networkx as nx

# =========================
# PREPARACIÓN
# =========================

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# =========================
# POLARIZACIÓN ROBUSTA
# =========================

# extracción segura (fallback automático)
df['polarization'] = df['sceneName'].str.extract(r'(VV|VH|HH|HV|SDV|SSV)')

df['polarization'] = df['polarization'].fillna('UNK')

# =========================
# AGRUPACIÓN MÁS FLEXIBLE
# =========================

groups = df.groupby([
    'pathNumber',
    'frameNumber',
    'flightDirection'
])

# =========================
# SBAS PARES
# =========================

pairs = []




for name, group in groups:

    group = group.sort_values('date').reset_index(drop=True)

    for i in range(len(group)):

        for j in range(i + 1, len(group)):

            t1 = group.loc[i, 'date']
            t2 = group.loc[j, 'date']

            baseline = (t2 - t1).days

            if baseline < MIN_BASELINE:
                continue

            if baseline > MAX_TEMPORAL_BASELINE:
                break

            pairs.append({
                'master': group.loc[i, 'sceneName'],
                'slave': group.loc[j, 'sceneName'],
                'master_date': t1,
                'slave_date': t2,
                'baseline_days': baseline,
                'pathNumber': group.loc[i, 'pathNumber'],
                'frameNumber': group.loc[i, 'frameNumber'],
                'direction': group.loc[i, 'flightDirection']
            })

pairs_df = pd.DataFrame(pairs)

# =========================
# VALIDACIÓN
# =========================

if pairs_df.empty:
    print("❌ No hay pares SBAS con estos parámetros")
    print("👉 Reduce MIN_BASELINE o revisa agrupación geométrica")
    raise SystemExit()

# =========================
# GRAFO SBAS
# =========================

G = nx.Graph()

for _, r in pairs_df.iterrows():

    G.add_edge(
        r['master'],
        r['slave'],
        weight=1 / r['baseline_days']
    )

mst = nx.maximum_spanning_tree(G)

# =========================
# ESCENAS OPTIMAS
# =========================

scenes = set()

for u, v in mst.edges():
    scenes.add(u)
    scenes.add(v)

scenes = sorted(scenes)

print("\n======================================")
print("SBAS OPTIMO (CORREGIDO)")
print("======================================")

print("Total pares:", len(pairs_df))
print("Total escenas:", len(scenes))

for s in scenes:
    print(s)

In [ ]:
import pandas as pd
import networkx as nx

# =============================================================================
# PREPARACIÓN
# =============================================================================

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)



# =============================================================================
# GRAFO GLOBAL SBAS
# =============================================================================

G = nx.Graph()

# =============================================================================
# CONSTRUCCIÓN DE RED POR GEOMETRÍA
# =============================================================================

for (path, frame, direction), group in df.groupby(['pathNumber','frameNumber','flightDirection']):

    group = group.sort_values('date').reset_index(drop=True)

    for i in range(len(group)):
        for j in range(i+1, len(group)):

            t1 = group.loc[i,'date']
            t2 = group.loc[j,'date']

            baseline = (t2 - t1).days

            if baseline < MIN_BASELINE:
                continue
            if baseline > MAX_TEMPORAL_BASELINE:
                break

            # peso: más corto = más fuerte
            weight = 1.0 / baseline

            G.add_edge(
                group.loc[i,'sceneName'],
                group.loc[j,'sceneName'],
                weight=weight,
                baseline=baseline
            )

# =============================================================================
# FILTRAR SOLO REDES CONECTADAS
# =============================================================================

components = list(nx.connected_components(G))

robust_scenes = set()

print("\n======================================")
print("RED SBAS ROBUSTA (PYGMTSAR STYLE)")
print("======================================\n")

for idx, comp in enumerate(components):

    subgraph = G.subgraph(comp)

    # extraer MST (red más fuerte dentro del componente)
    mst = nx.maximum_spanning_tree(subgraph)

    scenes = set()

    for u, v in mst.edges():
        scenes.add(u)
        scenes.add(v)

    robust_scenes.update(scenes)

    print(f"Componente {idx+1}")
    print(f"Escenas: {len(scenes)}")
    print("-----")

# =============================================================================
# RESULTADO FINAL
# =============================================================================

robust_scenes = sorted(list(robust_scenes))

print("\n======================================")
print("TOTAL RED ROBUSTA FINAL")
print("======================================")

print("Total escenas:", len(robust_scenes))

for s in robust_scenes:
    print(s)

In [ ]:
# =========================
# NORMALIZAR DATOS
# =========================

df = df.copy()

df['date'] = pd.to_datetime(df['date'])

df['flightDirection'] = (
    df['flightDirection']
    .astype(str)
    .str.upper()
    .str.strip()
)

# =========================
# FUNCIÓN SBAS GLOBAL
# =========================

def build_sbas(df_input, orbit_mode="ALL"):

    data = df_input.copy()

    # filtro seguro
    if orbit_mode != "ALL":
        data = data[data['flightDirection'] == orbit_mode].copy()

    if data.empty:
        print(f"⚠️ No hay datos para modo: {orbit_mode}")
        return pd.DataFrame()

    pairs = []

    # agrupar por geometría real
    for key, group in data.groupby(['pathNumber', 'frameNumber', 'flightDirection']):

        group = group.sort_values('date').reset_index(drop=True)

        # SBAS cadena temporal
        for i in range(len(group) - 1):

            master = group.iloc[i]
            slave  = group.iloc[i + 1]

            dt = (slave['date'] - master['date']).days

            if 3 <= dt <= MAX_TEMPORAL_BASELINE:

                pairs.append({
                    'master': master['sceneName'],
                    'slave': slave['sceneName'],
                    'master_date': master['date'],
                    'slave_date': slave['date'],
                    'days': dt,
                    'pathNumber': master['pathNumber'],
                    'frameNumber': master['frameNumber'],
                    'direction': master['flightDirection']
                })

    return pd.DataFrame(pairs)

# =========================
# EJECUTAR (CONTROLADO)
# =========================

ORBIT_MODE = "ASCENDING"   # o "DESCENDING"  "ASCENDING"   o "ALL"


pairs_df = build_sbas(df, ORBIT_MODE)

# =========================
# RESULTADO
# =========================

print("\n======================================")
print("SBAS ROBUSTO SIN ROMPER GEOMETRÍA")
print("======================================")

if pairs_df.empty:
    print("❌ No se generaron pares. Revisar ORBIT_MODE o datos.")
else:
    print(pairs_df[[
        'master','slave','days','pathNumber','frameNumber','direction'
    ]])

print("\nTotal pares:", len(pairs_df))

In [ ]:
from pygmtsar import S1, Stack
import matplotlib.pyplot as plt

for swath in [1, 2, 3]:

    print("\n======================================")
    print(f"ANALIZANDO SUBSWATH IW{swath}")
    print("======================================")

    try:
        scenes_swath = S1.scan_slc(
            DATADIR,
            subswath=swath,
            orbit='D'
        )
    except Exception as e:
        print(f"No se pudo escanear IW{swath}: {e}")
        continue

    print(f"Escenas encontradas IW{swath}: {len(scenes_swath)}")

    if scenes_swath.empty:
        continue

    fig, ax = plt.subplots(figsize=(8, 8))

    scenes_swath.boundary.plot(
        ax=ax,
        color='purple',
        alpha=0.6
    )

    aoi.boundary.plot(
        ax=ax,
        color='blue',
        linewidth=2,
        label='AOI'
    )

    ax.set_title(f"Sentinel-1 IW{swath} + AOI")
    ax.set_aspect('equal')
    ax.legend()
    plt.show()

In [ ]:
# =============================================================================
# SBAS DESCARGA ROBUSTA (SIN MEZCLA GEOMÉTRICA)
# =============================================================================

from pygmtsar import ASF

SUBSWATH = 2  # IW1=1, IW2=2, IW3=3

print("\n======================================")
print("SBAS - VALIDACIÓN FINAL ANTES DE DESCARGA")
print("======================================")

# =============================================================================
# 1. ASEGURAR COLUMNAS REQUERIDAS
# =============================================================================

required_cols = ['master', 'slave', 'pathNumber', 'frameNumber', 'direction']

for col in required_cols:
    if col not in pairs_df.columns:
        raise Exception(f"Falta columna crítica: {col}")

# =============================================================================
# 2. FILTRAR UNA SOLA GEOMETRÍA (CLAVE SBAS)
# =============================================================================


# No elegir un solo frame. Usar todos los frames válidos de pairs_df.
pairs_geo = pairs_df.copy()

download_scenes = sorted(
    set(pairs_geo['master'].tolist() + pairs_geo['slave'].tolist())
)

print(f"\nEscenas únicas SBAS para descargar: {len(download_scenes)}")

print("\nFrames incluidos:")
print(
    pairs_geo[['pathNumber', 'frameNumber', 'direction']]
    .drop_duplicates()
    .sort_values(['pathNumber', 'frameNumber'])
)
print(f"\nPares en geometría consistente: {len(pairs_geo)}")

if pairs_geo.empty:
    raise Exception("No hay pares en geometría consistente")

# =============================================================================
# 3. EXTRAER ESCENAS ÚNICAS
# =============================================================================

download_scenes = sorted(
    set(pairs_geo['master'].tolist() + pairs_geo['slave'].tolist())
)

print(f"\nEscenas únicas SBAS: {len(download_scenes)}")

# =============================================================================
# 4. VERIFICACIÓN FINAL DE CONSISTENCIA
# =============================================================================

print("\nVerificación rápida:")

print("Paths únicos   :", pairs_geo['pathNumber'].nunique())
print("Frames únicos  :", pairs_geo['frameNumber'].nunique())
print("Direcciones    :", pairs_geo['direction'].unique())

# Debe mantenerse una sola órbita/dirección, pero puede haber varios frames
if not (
    pairs_geo['pathNumber'].nunique() == 1 and
    pairs_geo['direction'].nunique() == 1
):
    raise Exception("❌ Mezcla de órbita o dirección detectada")

print("✔ SBAS consistente: misma órbita/dirección, múltiples frames permitidos")


In [ ]:
Dep = "Sechura"
# =============================================================================
# EXPORTAR TRÍADAS SBAS
# =============================================================================

# Crear diccionario Scene -> URL
scene_url = {
    row['sceneName']: row['url']
    for _, row in df.iterrows()
}

# Agregar URLs de descarga
triads_export = triads_df.copy()

triads_export['A_url'] = triads_export['A'].map(scene_url)
triads_export['B_url'] = triads_export['B'].map(scene_url)
triads_export['C_url'] = triads_export['C'].map(scene_url)

# Ruta de salida
CSV_TRIADAS = f'/content/drive/MyDrive/Inundacion_GEE/triadas_SBAS_{Dep}.csv'

# Exportar
triads_export.to_csv(
    CSV_TRIADAS,
    index=False
)

print("\n======================================")
print("TRÍADAS SBAS EXPORTADAS")
print("======================================")

print(f"Archivo : {CSV_TRIADAS}")
print(f"Total tríadas : {len(triads_export)}")

print("\nColumnas exportadas:")
print(list(triads_export.columns))

In [ ]:
!pip -q install --upgrade certifi requests urllib3 pyopenssl cryptography

import os, certifi, ssl

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

print("certifi:", certifi.where())
print("OpenSSL:", ssl.OPENSSL_VERSION)

In [ ]:
!pip -q install requests==2.32.4

In [ ]:
import requests, certifi

scene = download_scenes[0]
url = f"https://datapool.asf.alaska.edu/SLC/SB/{scene}.zip"

r = requests.get(url, timeout=30, stream=True, verify=certifi.where())
print(r.status_code)
print(r.url)

In [ ]:
from pygmtsar import ASF

asf_client = ASF(USERNAME, PASSWORD)

result = asf_client.download(
    DATADIR,
    download_scenes,
    SUBSWATH,
    n_jobs=1,
    retries=5,
    timeout_second=180,
    debug=True
)

print(result)


In [ ]:
# =============================================================================
# DESCARGAR ÓRBITAS PRECISAS (ROBUSTO SIN ERRORES)
# =============================================================================

from pygmtsar import S1

orbit_configs = {
    'D': 'DESCENDING',
    'A': 'ASCENDING'
}

print("\n======================================")
print("INICIANDO DESCARGA DE ÓRBITAS")
print("======================================")

# =============================================================================
# DETECTAR ORBITAS DISPONIBLES EN DATSET
# =============================================================================

available_orbits = df['flightDirection'].str.upper().unique()
print("\nÓRBITAS DISPONIBLES EN DATASET:", available_orbits)

# =============================================================================
# LOOP SEGURO
# =============================================================================

for orbit_code, orbit_name in orbit_configs.items():

    print('\n======================================')
    print(f'PROCESANDO ÓRBITA {orbit_name}')
    print('======================================')

    # ---------------------------------------------------------
    # ✔ VALIDACIÓN PREVIA (EVITA ERROR DE ASSERT)
    # ---------------------------------------------------------

    if orbit_name not in available_orbits:
        print(f"⚠️ No existe órbita {orbit_name} en el dataset")
        print("✔ Se omite sin error")
        continue

    # ---------------------------------------------------------
    # SCAN SEGURO
    # ---------------------------------------------------------

    try:
        scenes = S1.scan_slc(
            DATADIR,
            subswath=SUBSWATH,
            orbit=orbit_code
        )
    except AssertionError:
        print(f"⚠️ No hay escenas para órbita {orbit_name}")
        continue

    if scenes.empty:
        print(f"⚠️ Dataset vacío para órbita {orbit_name}")
        continue

    print('\nEscenas encontradas:\n')
    print(scenes[['datetime', 'orbit', 'orbitpath']])

    # ---------------------------------------------------------
    # DESCARGA ORBITAS
    # ---------------------------------------------------------

    print('\nDescargando órbitas...\n')

    try:
        S1.download_orbits(DATADIR, scenes)
        print(f"✔ Órbitas {orbit_name} descargadas correctamente")

    except Exception as e:
        print(f"❌ Error descargando órbitas {orbit_name}")
        print(e)

# =============================================================================
# VERIFICACIÓN FINAL
# =============================================================================

print('\n======================================')
print('VERIFICACIÓN FINAL')
print('======================================')

for orbit_code, orbit_name in orbit_configs.items():

    if orbit_name not in available_orbits:
        print(f"\n⚠️ {orbit_name} no fue procesada (no existe)")
        continue

    print('\n--------------------------------------')
    print(f'ÓRBITA {orbit_name}')
    print('--------------------------------------')

    try:
        scenes_check = S1.scan_slc(
            DATADIR,
            subswath=SUBSWATH,
            orbit=orbit_code
        )
    except AssertionError:
        print("⚠️ Sin datos")
        continue

    if scenes_check.empty:
        print("⚠️ Sin escenas")
        continue

    print(scenes_check[['datetime', 'orbit', 'orbitpath']])

    missing_orbits = scenes_check[scenes_check['orbitpath'].isna()]

    print('\nResumen:')
    print(f'Total escenas : {len(scenes_check)}')
    print(f'Sin órbita    : {len(missing_orbits)}')

    if len(missing_orbits) == 0:
        print('✔ Todas las órbitas están disponibles')
    else:
        print('⚠️ Faltan algunas órbitas')

In [ ]:
# =============================================================================
# DEM
# =============================================================================

DEM = f'{DATADIR}/dem.nc'

# =============================================================================
# DESCARGAR DEM COPERNICUS
# =============================================================================

print('\nDescargando DEM Copernicus...\n')

dem = Tiles().download_dem(
    aoi,
    filename=DEM
)

# =============================================================================
# VISUALIZAR DEM
# =============================================================================

dem.plot.imshow(
    cmap='cividis'
)

In [ ]:
# =============================================================================
# DASK (INICIALIZACIÓN LIMPIA)
# =============================================================================

import dask
from dask.distributed import Client

if 'client' in globals():
    client.close()

client = Client()
print(client)

# =============================================================================
# IMPORTS PYGMTSAR
# =============================================================================

from pygmtsar import S1, Stack

# =============================================================================
# ESCANEAR TODAS LAS ESCENAS SLC DESCENDENTES
# =============================================================================

print('\nEscaneando escenas SLC...\n')

scenes = S1.scan_slc(
    DATADIR,
    subswath=SUBSWATH,
    orbit='D'
)

print(scenes)

print(f'\nTotal escenas encontradas: {len(scenes)}')

# =============================================================================
# CREAR STACK CON LOS 6 FRAMES
# =============================================================================

print('\nCreando Stack InSAR...\n')

from shapely.geometry import MultiPolygon, Polygon

sbas = Stack(
    WORKDIR,
    drop_if_exists=True
).set_scenes(scenes)

print('\n======================================')
print('REFRAME DE ESCENAS')
print('======================================')


sbas.compute_reframe()

print('\n======================================')
print('STACK DATAFRAME')
print('======================================\n')

stack_df = sbas.to_dataframe()

print(stack_df)
print(f'\nNúmero de adquisiciones finales: {len(stack_df)}')

sbas.plot_scenes(aspect='equal')

In [ ]:
# =============================================================================
# MOSTRAR INFORMACIÓN
# =============================================================================

print('\n======================================')
print('RESUMEN STACK')
print('======================================')

print(f'Total escenas : {len(stack_df)}')
print(f'SUBSWATH      : IW{SUBSWATH}')
print(f'WORKDIR       : {WORKDIR}')
print(f'DATADIR       : {DATADIR}')


# =============================================================================
# VISUALIZAR FOOTPRINTS
# =============================================================================

import matplotlib.pyplot as plt

print('\nGraficando escenas...\n')

ax = sbas.plot_scenes(
    aspect='equal'
)


# =============================================================================
# GENERAR TRÍADAS DESDE EL STACK ACTUAL
# =============================================================================

print('\n======================================')
print('TRÍADAS DEL STACK ACTUAL')
print('======================================\n')


# obtener fechas únicas del stack reframeado
# Usar la columna 'datetime' para obtener los timestamps completos
dates_unique = stack_df['datetime'].unique()
dates = pd.Series(dates_unique).sort_values().reset_index(drop=True)


triads = []

for i in range(len(dates)-2):

    A = dates[i]
    B = dates[i+1]
    C = dates[i+2]

    triads.append({
        "A_date": A,
        "B_date": B,
        "C_date": C,
        "AB_days": (B-A).days,
        "BC_days": (C-B).days
    })


triads_df = pd.DataFrame(triads)



# =============================================================================
# MOSTRAR TRÍADAS
# =============================================================================

for idx,row in triads_df.iterrows():

    print('--------------------------------------')

    print(f'Tríada : {idx+1}\n')

    print(f'A : {row["A_date"]}')
    print(f'B : {row["B_date"]}')
    print(f'C : {row["C_date"]}\n')

    print(f'A-B : {row["AB_days"]} días')
    print(f'B-C : {row["BC_days"]} días')


    # metadata del stack
    # Buscar la fila en stack_df que coincida con el 'datetime' completo
    meta_row = stack_df[stack_df['datetime'] == row["A_date"]]
    if not meta_row.empty:
        meta = meta_row.iloc[0] # Obtener la primera fila coincidente
    else:
        print(f"Warning: No se encontraron metadatos en stack_df para el timestamp exacto {row['A_date']}")
        continue # Saltar la impresión de metadatos si no se encuentran

    print('\nPath      : revisar metadata')
    print(f'Mission   : {meta["mission"]}')
    print(f'Orbit     : {meta["orbit"]}')
    print(f'Subswath  : IW{meta["subswath"]}')

    print('\n--------------------------------------\n')


# =============================================================================
# FINAL
# =============================================================================

print('\n======================================')
print('STACK DESCENDING LISTO')
print('======================================')

In [ ]:
import matplotlib.pyplot as plt

print('\nMostrando Footprints + AOI + Mosaico C\n')


# Crear figura usando el plot de escenas
# Se crea la figura y los ejes explícitamente para controlar figsize
fig, ax = plt.subplots(figsize=(12,10))

# Ahora, sbas.plot_scenes() debería dibujar en estos ejes o en una nueva figura
# y no se le pasa figsize directamente
ax = sbas.plot_scenes(
    aspect='equal',
    ax=ax # Pasar los ejes para que pygmtsar los use si es compatible
)

# Si plot_scenes retorna None (aunque no debería si se le pasan los ejes)
# y queremos usar los ejes creados, podríamos hacer:
if ax is None:
    ax = plt.gca()

# =====================================================
# AGREGAR AOI
# =====================================================

try:

    # si es GeoDataFrame / GeoSeries
    aoi.boundary.plot(
        ax=ax,
        linewidth=2,
        label='AOI'
    )

except Exception as e:
    print("AOI no pudo graficarse como vector:", e)



# =====================================================
# CONFIGURACIÓN FINAL
# =====================================================

ax.set_title(
    f'Sentinel-1 Stack IW{SUBSWATH}\nFootprints + AOI ',
    fontsize=14
)


ax.legend()


plt.tight_layout()
plt.show()

In [ ]:
sbas.plot_scenes(aspect='equal')

In [ ]:
# define un AOI para acelerar el procesamiento
sbas.load_dem(DEM, aoi)

In [ ]:
sbas.plot_scenes(aspect='equal')

In [ ]:
sbas.compute_align()

In [ ]:
# =============================================================================
# COMPUTAR GEOCODIFICACIÓN
# =============================================================================

sbas.compute_geocode(45.)

In [ ]:
# =============================================================================
# DEFINIR PARES PARA LAS TRÍADAS DETECTADAS
# =============================================================================

import numpy as np

# =============================================================================
# LISTA DE PARES
# =============================================================================

pairs_list = []

# =============================================================================
# RECORRER TRÍADAS
# =============================================================================

for _, row in triads_df.iterrows():

    # -------------------------------------------------------------
    # FECHAS
    # -------------------------------------------------------------

    A_date = pd.to_datetime(row['A_date'])
    B_date = pd.to_datetime(row['B_date'])
    C_date = pd.to_datetime(row['C_date'])

    # -------------------------------------------------------------
    # PAR A-B
    # -------------------------------------------------------------

    pairs_list.append([
        A_date,
        B_date
    ])

    # -------------------------------------------------------------
    # PAR B-C
    # -------------------------------------------------------------

    pairs_list.append([
        B_date,
        C_date
    ])

# =============================================================================
# ARRAY FINAL
# =============================================================================

pairs = np.asarray(pairs_list)

# =============================================================================
# MOSTRAR TRÍADAS Y SUS PARES
# =============================================================================

print('\n======================================')
print('TRÍADAS Y PARES DE CORRELACIÓN')
print('======================================')

for idx, row in triads_df.iterrows():

    # -------------------------------------------------------------
    # FECHAS
    # -------------------------------------------------------------

    A_date = pd.to_datetime(row['A_date'])
    B_date = pd.to_datetime(row['B_date'])
    C_date = pd.to_datetime(row['C_date'])

    # -------------------------------------------------------------
    # BASELINES
    # -------------------------------------------------------------

    AB_days = abs((B_date - A_date).days)
    BC_days = abs((C_date - B_date).days)

    # -------------------------------------------------------------
    # PRINT
    # -------------------------------------------------------------

    print('\n--------------------------------------')
    print(f'TRÍADA {idx + 1}')
    print('--------------------------------------')

    print(f'A : {A_date}')
    print(f'B : {B_date}')
    print(f'C : {C_date}')

    print('\nPARES INTERFEROMÉTRICOS')

    print(f'  A-B : {AB_days} días')
    print(f'  B-C : {BC_days} días')

# =============================================================================
# RESUMEN
# =============================================================================

print('\n======================================')
print('RESUMEN')
print('======================================')

print(f'Total tríadas           : {len(triads_df)}')
print(f'Total pares derivados   : {len(pairs)}')

In [ ]:
# =============================================================================
# BLOQUE 1 — TOPOGRAFÍA Y DATOS SAR (CORREGIDO)
# =============================================================================

print('''
======================================
CARGANDO TOPOGRAFÍA Y DATOS SAR
======================================
''')

# -------------------------------------------------------------------------
# TOPOGRAFÍA
# -------------------------------------------------------------------------

topo = sbas.get_topo()
print(topo)

# -------------------------------------------------------------------------
# DATOS S1
# -------------------------------------------------------------------------

data = sbas.open_data()
print(data)

# -------------------------------------------------------------------------
# MULTILOOKING (AJUSTADO PARA CONSISTENCIA)
# -------------------------------------------------------------------------

print('\nCalculando intensidad multilooking...\n')

intensity = sbas.multilooking(
    np.square(np.abs(data)),
    wavelength=4,
    coarsen=COARSEN
)

print(intensity)

In [ ]:
# =============================================================================
# PARES INTERFEROMÉTRICOS DESDE STACK FINAL
# =============================================================================

dates = (
    pd.to_datetime(stack_df['datetime'])
    .sort_values()
    .reset_index(drop=True)
)

pairs = []

for i in range(len(dates) - 1):
    master = dates.iloc[i]
    slave = dates.iloc[i + 1]

    pairs.append([master, slave])

print("""
======================================
PARES INTERFEROMÉTRICOS
======================================
""")

if len(pairs) == 0:
    raise Exception("❌ No hay pares para procesar. Revisa stack_df.")

for idx, pair in enumerate(pairs):
    master = pd.to_datetime(pair[0])
    slave  = pd.to_datetime(pair[1])

    baseline = abs((slave - master).days)

    print(f"Par {idx + 1}")
    print(f"Master : {master}")
    print(f"Slave  : {slave}")
    print(f"Baseline temporal : {baseline} días")
    print("--------------------------------------")

In [ ]:
# =============================================================================
# BLOQUE 4 — DIFERENCIA DE FASE Y COHERENCIA (CORREGIDO)
# =============================================================================

import numpy as np
import pandas as pd
import dask
from pygmtsar import tqdm_dask

print("""
======================================
PROCESAMIENTO DE TRÍADAS (STACK ACTIVO)
======================================
""")

# =============================================================================
# CREAR TRÍADAS DESDE EL STACK FINAL
# =============================================================================

dates = (
    pd.to_datetime(stack_df['datetime'])
    .sort_values()
    .reset_index(drop=True)
)

triads = []

for i in range(len(dates) - 2):
    A = dates.iloc[i]
    B = dates.iloc[i + 1]
    C = dates.iloc[i + 2]

    triads.append({
        'A_date': A,
        'B_date': B,
        'C_date': C,
        'AB_days': abs((B - A).days),
        'BC_days': abs((C - B).days)
    })

triads_valid = pd.DataFrame(triads)

print("Tríadas generadas:")
print(triads_valid)

if triads_valid.empty:
    raise Exception("❌ No hay tríadas para procesar. Se necesitan al menos 3 fechas en stack_df.")

print(f"\nTríadas a procesar: {len(triads_valid)}")

# =============================================================================
# PARÁMETROS
# =============================================================================

if 'COARSEN' not in globals():
    COARSEN = (4, 4)

WAVELENGTH = 4

# =============================================================================
# INTENSIDAD MULTILOOKING
# =============================================================================

print("\nCalculando intensidad multilooking...\n")

intensity = sbas.multilooking(
    np.square(np.abs(data)),
    wavelength=WAVELENGTH,
    coarsen=COARSEN
)

# =============================================================================
# LISTAS DE CONTROL
# =============================================================================

processed_triads = []
skipped_triads = []

# Guardan resultados para exportar después
phase_results = {}
corr_results = {}

# =============================================================================
# LOOP PRINCIPAL
# =============================================================================

for i, row in triads_valid.iterrows():

    print("\n======================================")
    print(f"TRÍADA {i + 1}")
    print("======================================")

    A_date = pd.to_datetime(row['A_date'])
    B_date = pd.to_datetime(row['B_date'])
    C_date = pd.to_datetime(row['C_date'])

    print(f"A: {A_date}")
    print(f"B: {B_date}")
    print(f"C: {C_date}")
    print(f"A-B: {row['AB_days']} días")
    print(f"B-C: {row['BC_days']} días")

    pairs = [
        [A_date.strftime('%Y-%m-%d'), B_date.strftime('%Y-%m-%d')],
        [B_date.strftime('%Y-%m-%d'), C_date.strftime('%Y-%m-%d')]
    ]

    print("\nPares para PyGMTSAR:")
    print(pairs)

    print("\nCalculando fase diferencial...")

    try:
        phase = sbas.phasediff(
            pairs,
            data,
            topo
        )

        phase_ml = sbas.multilooking(
            phase,
            wavelength=WAVELENGTH,
            coarsen=COARSEN
        )

        # Guardar interferogramas/fase multilooked para exportar después
        phase_results[f"triad_{i + 1}"] = phase_ml

        print("Calculando coherencia...")

        corr = sbas.correlation(
            phase_ml,
            intensity
        )

        corr = dask.persist(corr)[0]

        tqdm_dask(
            corr,
            desc=f"Triad {i + 1}"
        )

        # Guardar coherencia/correlación para exportar después
        corr_results[f"triad_{i + 1}"] = corr

        print("✔ Coherencia calculada")

        processed_triads.append({
            "triad": i + 1,
            "A": A_date,
            "B": B_date,
            "C": C_date
        })

    except Exception as e:

        print("❌ ERROR EN TRÍADA:", str(e))

        skipped_triads.append({
            "triad": i + 1,
            "error": str(e)
        })

# =============================================================================
# RESUMEN FINAL
# =============================================================================

print("""
======================================
RESUMEN FINAL
======================================
""")

print(f"Procesadas: {len(processed_triads)}")
print(f"Omitidas  : {len(skipped_triads)}")

if skipped_triads:
    print("\nTríadas omitidas:")
    for item in skipped_triads:
        print(item)

print("\n✔ PROCESAMIENTO TERMINADO")

In [ ]:
corr_ll = sbas.ra2ll(corr)

sbas.plot_correlations(
    corr_ll.where(corr_ll<0.2),
    cols=2,
    cmap='turbo',
    caption='Correlation Lost: Indicates Flooding'
)

In [ ]:
# =============================================================================
# EXPORTAR INTERFEROGRAMAS Y COHERENCIA A GEOTIFF
# =============================================================================

import os
import numpy as np
import rioxarray

outdir = "/content/sample_data/exports_insar"
os.makedirs(outdir, exist_ok=True)

print("phase_results:", list(phase_results.keys()) if 'phase_results' in globals() else "NO EXISTE")
print("corr_results :", list(corr_results.keys()) if 'corr_results' in globals() else "NO EXISTE")

if 'phase_results' not in globals() or len(phase_results) == 0:
    raise Exception("❌ phase_results está vacío. Ejecuta primero el BLOQUE 4 y verifica que procese al menos 1 tríada.")

if 'corr_results' not in globals() or len(corr_results) == 0:
    raise Exception("❌ corr_results está vacío. Ejecuta primero el BLOQUE 4 y verifica que procese al menos 1 tríada.")


def prepare_for_geotiff(da, is_phase=False):
    da = da.squeeze()

    # Si la fase/interferograma es complejo, exportar ángulo de fase
    if np.iscomplexobj(da.dtype):
        if is_phase:
            da = np.angle(da)
        else:
            da = np.abs(da)

    da = da.astype("float32")

    rename_dims = {}
    if "lat" in da.dims:
        rename_dims["lat"] = "y"
    if "lon" in da.dims:
        rename_dims["lon"] = "x"

    if rename_dims:
        da = da.rename(rename_dims)

    if "x" not in da.dims or "y" not in da.dims:
        raise Exception(f"❌ No encuentro dimensiones espaciales x/y. Dims actuales: {da.dims}")

    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

    # Si x/y son lon/lat reales, escribir EPSG:4326
    da = da.rio.write_crs("EPSG:4326", inplace=False)

    return da


for triad_name in corr_results.keys():

    print("\n======================================")
    print(f"EXPORTANDO {triad_name}")
    print("======================================")

    corr_da = corr_results[triad_name]
    phase_da = phase_results[triad_name]

    triad_index = int(triad_name.split("_")[1]) - 1

    A_date = pd.to_datetime(triads_valid.loc[triad_index, "A_date"])
    B_date = pd.to_datetime(triads_valid.loc[triad_index, "B_date"])
    C_date = pd.to_datetime(triads_valid.loc[triad_index, "C_date"])

    pair_names = [
        f"{A_date.strftime('%Y%m%d')}_{B_date.strftime('%Y%m%d')}",
        f"{B_date.strftime('%Y%m%d')}_{C_date.strftime('%Y%m%d')}"
    ]

    print("Dims phase:", phase_da.dims)
    print("Dims corr :", corr_da.dims)

    for pair_idx, pair_name in enumerate(pair_names):

        phase_pair = phase_da.isel(pair=pair_idx).compute()
        phase_pair = prepare_for_geotiff(phase_pair, is_phase=True)

        phase_file = os.path.join(outdir, f"Interferograma_Fase_{pair_name}.tif")
        phase_pair.rio.to_raster(phase_file)

        print(f"✔ Exportado: {phase_file}")

        corr_pair = corr_da.isel(pair=pair_idx).compute()
        corr_pair = prepare_for_geotiff(corr_pair, is_phase=False)

        corr_file = os.path.join(outdir, f"Coherencia_{pair_name}.tif")
        corr_pair.rio.to_raster(corr_file)

        print(f"✔ Exportado: {corr_file}")


print("\n======================================")
print("EXPORTACIÓN FINALIZADA")
print("======================================")
print(f"Archivos guardados en: {outdir}")

print("\nArchivos creados:")
print(os.listdir(outdir))

In [ ]:
# =============================================================================
# DIAGNÓSTICO BLOQUE 4 — SIN TRY/EXCEPT
# =============================================================================

import numpy as np
import pandas as pd
import dask
from pygmtsar import tqdm_dask

if 'COARSEN' not in globals():
    COARSEN = (4, 4)

WAVELENGTH = 4

dates = (
    pd.to_datetime(stack_df['datetime'])
    .sort_values()
    .reset_index(drop=True)
)

A_date = dates.iloc[0]
B_date = dates.iloc[1]
C_date = dates.iloc[2]

pairs = [
    [A_date.strftime('%Y-%m-%d'), B_date.strftime('%Y-%m-%d')],
    [B_date.strftime('%Y-%m-%d'), C_date.strftime('%Y-%m-%d')]
]

print("Pairs:")
print(pairs)

print("\nData:")
print(data)

print("\nTopo:")
print(topo)

print("\nCalculando intensity...")
intensity_test = sbas.multilooking(
    np.square(np.abs(data)),
    wavelength=WAVELENGTH,
    coarsen=COARSEN
)
print(intensity_test)

print("\nCalculando phase...")
phase_test = sbas.phasediff(
    pairs,
    data,
    topo
)
print(phase_test)

print("\nCalculando phase_ml...")
phase_ml_test = sbas.multilooking(
    phase_test,
    wavelength=WAVELENGTH,
    coarsen=COARSEN
)
print(phase_ml_test)

print("\nCalculando corr...")
corr_test = sbas.correlation(
    phase_ml_test,
    intensity_test
)
print(corr_test)

print("\nEjecutando Dask...")
corr_test = dask.persist(corr_test)[0]
tqdm_dask(corr_test, desc="corr_test")

print("\n✔ Diagnóstico terminado correctamente")

phase_results = {
    "triad_1": phase_ml_test
}

corr_results = {
    "triad_1": corr_test
}

triads_valid = pd.DataFrame([{
    "A_date": A_date,
    "B_date": B_date,
    "C_date": C_date,
    "AB_days": abs((B_date - A_date).days),
    "BC_days": abs((C_date - B_date).days)
}])

print("phase_results:", phase_results.keys())
print("corr_results :", corr_results.keys())

In [ ]:
phase_results = {}
corr_results = {}

In [ ]:
# =============================================================================
# RECALCULAR Y EXPORTAR INTERFEROGRAMAS, COHERENCIA Y CORRELACIÓN A GEOTIFF
# =============================================================================

import os
import numpy as np
import pandas as pd
import xarray as xr
import dask
import rioxarray
from pygmtsar import tqdm_dask

# =============================================================================
# PARÁMETROS
# =============================================================================

outdir = "/content/sample_data/exports_insar"
os.makedirs(outdir, exist_ok=True)

if 'COARSEN' not in globals():
    COARSEN = (4, 4)

WAVELENGTH = 4

# =============================================================================
# FECHAS Y PARES
# =============================================================================

dates = (
    pd.to_datetime(stack_df['datetime'])
    .sort_values()
    .reset_index(drop=True)
)

if len(dates) < 3:
    raise Exception("❌ Se necesitan al menos 3 fechas para formar la tríada.")

A_date = dates.iloc[0]
B_date = dates.iloc[1]
C_date = dates.iloc[2]

pairs = [
    [A_date.strftime('%Y-%m-%d'), B_date.strftime('%Y-%m-%d')],
    [B_date.strftime('%Y-%m-%d'), C_date.strftime('%Y-%m-%d')]
]

pair_names = [
    f"{A_date.strftime('%Y%m%d')}_{B_date.strftime('%Y%m%d')}",
    f"{B_date.strftime('%Y%m%d')}_{C_date.strftime('%Y%m%d')}"
]

print("Pares:")
print(pairs)

# =============================================================================
# RECALCULAR INTERFEROGRAMA Y COHERENCIA
# =============================================================================

print("\nCalculando intensidad multilooking...")

intensity = sbas.multilooking(
    np.square(np.abs(data)),
    wavelength=WAVELENGTH,
    coarsen=COARSEN
)

print("\nCalculando fase diferencial...")

phase = sbas.phasediff(
    pairs,
    data,
    topo
)

print("\nCalculando fase multilooking...")

phase_ml = sbas.multilooking(
    phase,
    wavelength=WAVELENGTH,
    coarsen=COARSEN
)

print("\nCalculando coherencia/correlación...")

corr = sbas.correlation(
    phase_ml,
    intensity
)

print("\nPersistiendo coherencia/correlación con Dask...")

corr = dask.persist(corr)[0]
tqdm_dask(corr, desc="Coherencia")

phase_results = {"triad_1": phase_ml}
corr_results = {"triad_1": corr}

print("phase_results:", list(phase_results.keys()))
print("corr_results :", list(corr_results.keys()))

# =============================================================================
# FUNCIONES DE EXPORTACIÓN
# =============================================================================

def phase_angle_dataarray(phase_da):
    return xr.apply_ufunc(
        np.angle,
        phase_da,
        dask="allowed"
    ).astype("float32")


def prepare_geocoded_for_geotiff(da):
    da = da.squeeze()

    rename_dims = {}

    if "lat" in da.dims:
        rename_dims["lat"] = "y"

    if "lon" in da.dims:
        rename_dims["lon"] = "x"

    if rename_dims:
        da = da.rename(rename_dims)

    if "x" not in da.dims or "y" not in da.dims:
        raise Exception(f"❌ No encuentro dimensiones x/y. Dims actuales: {da.dims}")

    da = da.astype("float32")
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    da = da.rio.write_crs("EPSG:4326", inplace=False)

    return da

# =============================================================================
# EXPORTAR
# =============================================================================

print("\n======================================")
print("EXPORTANDO GEOTIFFS")
print("======================================")

for pair_idx, pair_name in enumerate(pair_names):

    # -------------------------------------------------------------------------
    # INTERFEROGRAMA / FASE
    # -------------------------------------------------------------------------

    print(f"\nGeocodificando interferograma {pair_name}...")

    phase_pair = phase_ml.isel(pair=pair_idx)
    phase_angle = phase_angle_dataarray(phase_pair)

    phase_ll = sbas.ra2ll(phase_angle).compute()
    phase_ll = prepare_geocoded_for_geotiff(phase_ll)

    phase_file = os.path.join(
        outdir,
        f"Interferograma_Fase_{pair_name}.tif"
    )

    phase_ll.rio.to_raster(phase_file)
    print(f"✔ Interferograma exportado: {phase_file}")

    # -------------------------------------------------------------------------
    # COHERENCIA
    # -------------------------------------------------------------------------

    print(f"Geocodificando coherencia {pair_name}...")

    coh_pair = corr.isel(pair=pair_idx).astype("float32")

    coh_ll = sbas.ra2ll(coh_pair).compute()
    coh_ll = prepare_geocoded_for_geotiff(coh_ll)

    coh_file = os.path.join(
        outdir,
        f"Coherencia_{pair_name}.tif"
    )

    coh_ll.rio.to_raster(coh_file)
    print(f"✔ Coherencia exportada: {coh_file}")

    # -------------------------------------------------------------------------
    # CORRELACIÓN
    # -------------------------------------------------------------------------

    corr_file = os.path.join(
        outdir,
        f"Correlacion_{pair_name}.tif"
    )

    coh_ll.rio.to_raster(corr_file)
    print(f"✔ Correlación exportada: {corr_file}")

# =============================================================================
# RESUMEN
# =============================================================================

print("\n======================================")
print("EXPORTACIÓN FINALIZADA")
print("======================================")
print(f"Archivos guardados en: {outdir}")
print(os.listdir(outdir))

In [ ]:
# =====================================================
# COMPRIMIR Y DESCARGAR data_kal + raw_kal
# =====================================================

from google.colab import files
import os

ZIP_NAME = "/content/PyGMTSAR_Proyecto.zip"

# Eliminar zip previo si existe
if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

# Crear ZIP
!zip -r "$ZIP_NAME" /content/data_kal

print(f"\nZIP creado: {ZIP_NAME}\n")

# Descargar al computador
files.download(ZIP_NAME)

In [ ]:
# =====================================================
# DESCARGAR TODOS LOS ARCHIVOS DE data_kal INDIVIDUALMENTE
# =====================================================

from google.colab import files
import os

CARPETA = "/content/raw_kal"

for root, dirs, files_list in os.walk(CARPETA):
    for archivo in files_list:
        ruta = os.path.join(root, archivo)
        print(f"Descargando: {ruta}")
        files.download(ruta)

In [ ]:
import os
import pandas as pd

carpeta = "/content/data_kal"

registros = []

for archivo in os.listdir(carpeta):
    if archivo.endswith(".SAFE"):
        escena = archivo.replace(".SAFE", "")

        sat = "SA" if escena.startswith("S1A") else "SB"

        registros.append({
            "Scene_ID": escena,
            "url": f"https://datapool.asf.alaska.edu/SLC/{sat}/{escena}.zip"
        })

df = pd.DataFrame(registros)
df.to_csv("/content/escenas_con_links.csv", index=False)

print(df.head())

In [ ]:
# =====================================================
# COMPRIMIR Y GUARDAR EN GOOGLE DRIVE
# =====================================================

from google.colab import drive
import os

drive.mount('/content/drive')

ZIP_NAME = "/content/drive/MyDrive/PyGMTSAR_Proyecto.zip"

if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

!zip -r "$ZIP_NAME" /content/data_kal /content/raw_kal

print(f"\nZIP guardado en:\n{ZIP_NAME}")

**anterior**

**Modelo base**